In [ ]:
from transformers import OneFormerProcessor, OneFormerForUniversalSegmentation, AutoImageProcessor, UperNetForSemanticSegmentation

from diffusers.utils import load_image, make_image_grid
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    StableDiffusionControlNetImg2ImgPipeline,
    UniPCMultistepScheduler,
    DPMSolverMultistepScheduler,
    KDPM2DiscreteScheduler,
    KDPM2AncestralDiscreteScheduler
)

from compel import Compel

from controlnet_aux import CannyDetector, MidasDetector
from controlnet_aux.util import resize_image

import torch
import pathlib
import numpy as np
import cv2

from IPython.display import display
import PIL.Image as Image

In [ ]:
seed = 0
prompt = "drones flying in the snowy weather, snowstorm"
negative_prompt = "lowres, sketches"

In [ ]:
def resize(image: Image.Image):
    return Image.fromarray(resize_image(np.array(image, dtype=np.uint8), 512))

root_path = pathlib.Path("/home/leafying/data/UAV/DUT_Anti_UAV/detection/images/val")
image_paths = sorted(root_path.rglob("*.jpg"))

init_images = [resize(load_image(image_path.as_posix())) for image_path in image_paths[:5]]
make_image_grid([resize(init_image) for init_image in init_images], rows=1, cols=len(init_images))

In [ ]:
canny = CannyDetector()
def detect_canny_edges(
    image: Image.Image, low_threshold: int = 100, high_threshold: int = 200
):
    return canny(image, low_threshold=low_threshold, high_threshold=high_threshold)

canny_images = [detect_canny_edges(init_image, low_threshold=100, high_threshold=200) for init_image in init_images]
make_image_grid(canny_images, rows=1, cols=len(canny_images))

In [ ]:
midas = MidasDetector.from_pretrained("lllyasviel/Annotators").to("cuda")
depth_images = [midas(init_image) for init_image in init_images]
make_image_grid(depth_images, rows=1, cols=len(depth_images))

In [ ]:
# processor = DPTImageProcessor.from_pretrained("Intel/dpt-large")
# model = DPTForDepthEstimation.from_pretrained("Intel/dpt-large")

# image = init_images[0]
# # image = np.array(image, dtype=np.uint8)
# # image = resize_image(image, 512)

# # print(image.shape)

# # prepare image for the model
# inputs = processor(images=image, return_tensors="pt")

# with torch.no_grad():
#     outputs = model(**inputs)
#     predicted_depth = outputs.predicted_depth

# # interpolate to original size
# prediction = torch.nn.functional.interpolate(
#     predicted_depth.unsqueeze(1),
#     size=image.size[::-1],
#     mode="bilinear",
#     align_corners=False,
# )

# # visualize the prediction
# output = prediction.squeeze().cpu().numpy()
# formatted = (output * 255 / np.max(output)).astype("uint8")
# depth = Image.fromarray(formatted)

# depth

In [ ]:
ADE20K_SEM_SEG_CATEGORIES = [
    "wall",
    "building",
    "sky",
    "floor",
    "tree",
    "ceiling",
    "road, route",
    "bed",
    "window ",
    "grass",
    "cabinet",
    "sidewalk, pavement",
    "person",
    "earth, ground",
    "door",
    "table",
    "mountain, mount",
    "plant",
    "curtain",
    "chair",
    "car",
    "water",
    "painting, picture",
    "sofa",
    "shelf",
    "house",
    "sea",
    "mirror",
    "rug",
    "field",
    "armchair",
    "seat",
    "fence",
    "desk",
    "rock, stone",
    "wardrobe, closet, press",
    "lamp",
    "tub",
    "rail",
    "cushion",
    "base, pedestal, stand",
    "box",
    "column, pillar",
    "signboard, sign",
    "chest of drawers, chest, bureau, dresser",
    "counter",
    "sand",
    "sink",
    "skyscraper",
    "fireplace",
    "refrigerator, icebox",
    "grandstand, covered stand",
    "path",
    "stairs",
    "runway",
    "case, display case, showcase, vitrine",
    "pool table, billiard table, snooker table",
    "pillow",
    "screen door, screen",
    "stairway, staircase",
    "river",
    "bridge, span",
    "bookcase",
    "blind, screen",
    "coffee table",
    "toilet, can, commode, crapper, pot, potty, stool, throne",
    "flower",
    "book",
    "hill",
    "bench",
    "countertop",
    "stove",
    "palm, palm tree",
    "kitchen island",
    "computer",
    "swivel chair",
    "boat",
    "bar",
    "arcade machine",
    "hovel, hut, hutch, shack, shanty",
    "bus",
    "towel",
    "light",
    "truck",
    "tower",
    "chandelier",
    "awning, sunshade, sunblind",
    "street lamp",
    "booth",
    "tv",
    "plane",
    "dirt track",
    "clothes",
    "pole",
    "land, ground, soil",
    "bannister, banister, balustrade, balusters, handrail",
    "escalator, moving staircase, moving stairway",
    "ottoman, pouf, pouffe, puff, hassock",
    "bottle",
    "buffet, counter, sideboard",
    "poster, posting, placard, notice, bill, card",
    "stage",
    "van",
    "ship",
    "fountain",
    "conveyer belt, conveyor belt, conveyer, conveyor, transporter",
    "canopy",
    "washer, automatic washer, washing machine",
    "plaything, toy",
    "pool",
    "stool",
    "barrel, cask",
    "basket, handbasket",
    "falls",
    "tent",
    "bag",
    "minibike, motorbike",
    "cradle",
    "oven",
    "ball",
    "food, solid food",
    "step, stair",
    "tank, storage tank",
    "trade name",
    "microwave",
    "pot",
    "animal",
    "bicycle",
    "lake",
    "dishwasher",
    "screen",
    "blanket, cover",
    "sculpture",
    "hood, exhaust hood",
    "sconce",
    "vase",
    "traffic light",
    "tray",
    "trash can",
    "fan",
    "pier",
    "crt screen",
    "plate",
    "monitor",
    "bulletin board",
    "shower",
    "radiator",
    "glass, drinking glass",
    "clock",
    "flag",  # noqa
]
len(ADE20K_SEM_SEG_CATEGORIES)

In [ ]:
from controlnet_aux.util import ade_palette

processor = OneFormerProcessor.from_pretrained("shi-labs/oneformer_ade20k_swin_large")
model = OneFormerForUniversalSegmentation.from_pretrained(
    "shi-labs/oneformer_ade20k_swin_large"
).to("cuda")

palette = [
    {"name": ADE20K_SEM_SEG_CATEGORIES[i], "color": color}
    for i, color in enumerate(ade_palette())
]
segment_images = []
for init_image, depth_image in zip(init_images, depth_images):
    inputs = processor(init_image, task_inputs=["semantic"], return_tensors="pt").to(
        "cuda"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    seg = (
        processor.post_process_semantic_segmentation(
            outputs, target_sizes=[depth_image.size[::-1]]
        )[0]
        .cpu()
        .numpy()
    )

    color_seg = np.zeros(
        (seg.shape[0], seg.shape[1], 3), dtype=np.uint8
    )  # height, width, 3
    for label, info in enumerate(palette):
        # if info["name"] in ("plane",):
        #     continue

        color = info["color"]
        color_seg[seg == label, :] = color

    segment_image = Image.fromarray(color_seg)
    segment_images.append(segment_image)
make_image_grid(segment_images, rows=1, cols=len(segment_images))

In [ ]:
from controlnet_aux.util import ade_palette

processor = AutoImageProcessor.from_pretrained("openmmlab/upernet-convnext-small")
model = UperNetForSemanticSegmentation.from_pretrained(
    "openmmlab/upernet-convnext-small"
).to("cuda")

palette = ade_palette()
segment_images = []
for init_image, depth_image in zip(init_images, depth_images):
    pixel_values = processor(init_image, return_tensors="pt").pixel_values.to("cuda")

    with torch.no_grad():
        outputs = model(pixel_values)

    seg = (
        processor.post_process_semantic_segmentation(
            outputs, target_sizes=[depth_image.size[::-1]]
        )[0]
        .cpu()
        .numpy()
    )

    color_seg = np.zeros(
        (seg.shape[0], seg.shape[1], 3), dtype=np.uint8
    )  # height, width, 3
    for label, color in enumerate(palette):
        color_seg[seg == label, :] = color

    segment_image = Image.fromarray(color_seg)
    segment_images.append(segment_image)
make_image_grid(segment_images, rows=1, cols=len(segment_images))

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"

# ControlNet 1.0

## Canny

In [ ]:
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16, use_safetensors=True)

### Text to Image

In [ ]:
pipeline = StableDiffusionControlNetPipeline.from_pretrained(
    model_id, controlnet=controlnet, torch_dtype=torch.float16, use_safetensors=True
)
pipeline.scheduler = UniPCMultistepScheduler.from_config(pipeline.scheduler.config)

pipeline.enable_model_cpu_offload()

In [ ]:
generator = torch.Generator().manual_seed(seed)
for init_image, canny_image in zip(init_images, canny_images):
    output = pipeline(
        prompt, image=canny_image, negative_prompt=negative_prompt, generator=generator
    ).images[0]
    print(output.size)
    display(make_image_grid([canny_image, output, init_image], rows=1, cols=3))

### Image to Image

## Depth

In [ ]:
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-depth", torch_dtype=torch.float16, use_safetensors=True)

### Text to Image

In [ ]:
pipeline = StableDiffusionControlNetPipeline.from_pretrained(
    model_id, controlnet=controlnet, torch_dtype=torch.float16, use_safetensors=True
)
pipeline.scheduler = UniPCMultistepScheduler.from_config(pipeline.scheduler.config)

pipeline.enable_model_cpu_offload()

In [ ]:
generator = torch.Generator().manual_seed(seed)
for init_image, depth_image in zip(init_images, depth_images):
    output = pipeline(prompt, image=depth_image, negative_prompt=negative_prompt, generator=generator).images[0]
    display(make_image_grid([depth_image, output, init_image], rows=1, cols=3))

## Canny + Depth

In [ ]:
controlnet_models = ["lllyasviel/sd-controlnet-canny", "lllyasviel/sd-controlnet-depth"]
controlnets = [
    ControlNetModel.from_pretrained(
        controlnet_model, torch_dtype=torch.float16, use_safetensors=True
    )
    for controlnet_model in controlnet_models
]

### Text to Image

In [ ]:
pipeline = StableDiffusionControlNetPipeline.from_pretrained(
    model_id, controlnet=controlnets, torch_dtype=torch.float16, use_safetensors=True
)
pipeline.scheduler = UniPCMultistepScheduler.from_config(pipeline.scheduler.config)

pipeline.enable_model_cpu_offload()

In [ ]:
generator = torch.Generator().manual_seed(seed)
for init_image, canny_image, depth_image in zip(
    init_images, canny_images, depth_images
):
    display(make_image_grid([canny_image, depth_image, init_image], rows=1, cols=3))
    output = pipeline(
        prompt,
        image=[canny_image, depth_image],
        negative_prompt=negative_prompt,
        controlnet_conditioning_scale=[1.0, 0.8],
        generator=generator,
    ).images[0]
    display(output)

## Canny + Depth + Segment

In [ ]:
controlnet_models = ["lllyasviel/control_v11p_sd15_canny", "lllyasviel/control_v11f1p_sd15_depth", "lllyasviel/control_v11p_sd15_seg"]
controlnets = [
    ControlNetModel.from_pretrained(
        controlnet_model, torch_dtype=torch.float16, use_safetensors=True
    )
    for controlnet_model in controlnet_models
]

In [ ]:
pipeline: StableDiffusionControlNetPipeline = (
    StableDiffusionControlNetPipeline.from_pretrained(
        model_id,
        controlnet=controlnets,
        torch_dtype=torch.float16,
        use_safetensors=True,
    )
)
# pipeline: StableDiffusionControlNetImg2ImgPipeline = (
#     StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
#         model_id,
#         controlnet=controlnets,
#         torch_dtype=torch.float16,
#         use_safetensors=True,
#     )
# )
# pipeline.scheduler = UniPCMultistepScheduler.from_config(pipeline.scheduler.config)

# DPM++ 2M Karras
# pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config, use_karras_sigmas=True)

# DPM++ 2M SDE Karras
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config, use_karras_sigmas=True, algorithm_type="sde-dpmsolver++")

# DPM2 Karras
# pipeline.scheduler = KDPM2DiscreteScheduler.from_config(
#     pipeline.scheduler.config, use_karras_sigmas=True
# )

# DPM2 a Karras
# pipeline.scheduler = KDPM2AncestralDiscreteScheduler.from_config(
#     pipeline.scheduler.config, use_karras_sigmas=True
# )

compel_proc = Compel(tokenizer=pipeline.tokenizer, text_encoder=pipeline.text_encoder)

pipeline.enable_model_cpu_offload()

In [ ]:
seed = 0

prompt_prefix = "A photograph of "
prompt_suffix = ", photorealistic, vivid, high resolution, 8k, highly detailed, Canon R6 Mark II, 35 mm lens."

prompt = (
    f"{prompt_prefix}drones flying in snowy weather, (visibility reduced to mere feet in the snowstorm)++{prompt_suffix}"
)
negative_prompt = "lowres, sketches, paintings"

generator = torch.Generator().manual_seed(seed)

In [ ]:
for init_image, canny_image, depth_image, segment_image in zip(
    init_images, canny_images, depth_images, segment_images
):
    display(
        make_image_grid(
            [canny_image, depth_image, segment_image, init_image], rows=1, cols=4
        )
    )
    output = pipeline(
        prompt_embeds=compel_proc(prompt),
        # image=init_image,
        image=[canny_image, depth_image, segment_image],
        negative_prompt_embeds=compel_proc(negative_prompt),
        controlnet_conditioning_scale=[0.3, 0.3, 0.3],
        generator=generator,
        num_inference_steps=50,
        # guidance_scale=8.0,
        # strength=0.9,
    ).images[0]
    display(output)

# ControlNet 1.1